# T11 It1: a candidate judge trained on satellite imagery only

**Rule (team, 2026-11-14):** learned components are trained on satellite imagery only; thermal photos (NUDT, official) may set cut thresholds
and test. **Training data:** 400 drone-view photos rendered from Cartosat scene B and Pleiades Neo (not the map), run through the unchanged
pipeline (`Sat_Photo_Dump`), every candidate labelled by its known true place. **Judges:** S1 = the T7 architecture (monotone GBM, 19 signals);
S5 = relative inliers + scale augmentation (T10 V5). **Cut:** Learn-then-Test (alpha 0.005, floor 0.9) on the NUDT design sets (101-606), with
P from the satellite-trained judge (thermal data only sets the number). **Check:** official TRY (and the NUDT design sets, which only set the
cut). Official TEST, NUDT 707 / 808 are seen data; NUDT 909 is the fresh final set (It2, after pre-registration).
Reference: the NUDT-trained judge (T10), which may not ship.

In [ ]:
import os, sys, time, gc
os.environ.setdefault("OMP_NUM_THREADS", "4")
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
import train.checks.candidate_judge as cj
import train.checks.scale_robust as sr
from train.checks.decision_signals import build_signal_table
from train.checks.holdout import NUDT_DESIGN, answer_change, bar_frontier, load_t8_table, versus_frontier
from train.checks.eval_tools import manifest, save_manifest
R = ROOT / "tests/Test_Iteration_11/results"; (R / "signals").mkdir(exist_ok=True)
out = R / "signals" / "sat_train.csv.gz"
if not out.exists():
    build_signal_table(R / "rich_dump" / "sat_train", "sat", 4).to_csv(out, index=False)
SAT = pd.read_csv(out)
items = pd.read_csv(R / "sat_photos" / "sat_train" / "items.csv")
from sklearn.cluster import KMeans
fold = KMeans(5, random_state=0, n_init=10).fit_predict(items[["east", "north"]].to_numpy())     # 5 groups of nearby places
SAT["data"] = ["sat%d" % fold[p] for p in SAT.photo]
SAT_SETS = sorted(SAT.data.unique())
SAT = cj.add_context(cj.add_features(SAT))
D = load_t8_table(ROOT, locked=True)                          # NUDT design + official TRY
print(SAT.photo.nunique(), "satellite-made photos;", SAT.groupby("data").photo.nunique().to_dict())

## 1. What the satellite-made candidates look like next to thermal ones (labelled candidates, >= 3 inliers)

In [ ]:
lab = lambda t: t[cj.labelled(t)].assign(right=lambda d: d.err_sigma <= 4)
cmp = pd.concat([lab(SAT).assign(src="satellite-made"), lab(D[D.data.isin(NUDT_DESIGN)]).assign(src="NUDT design"), lab(D[D.data == "official_try"]).assign(src="official TRY")])
cmp.groupby(["src", "right"])[["inliers", "z", "hull", "cover9", "zgap", "ligap", "angle", "shift_px"]].median().round(3)

## 2. Judges trained on satellite-made candidates only; cut on the NUDT design sets

In [ ]:
F = cj.FEATURES_CTX; FR = [("rli" if f == "li" else "rln22" if f == "ln22" else f) for f in F]; AUG = [0.5, 0.75, 1.0, 1.5, 2.0]
def judges(m):
    sat = SAT if m is None else cj.add_context(SAT, m=m)
    ev = D if m is None else cj.add_context(D, m=m)
    J = {}
    _, s1 = cj.judge_probability(sat, SAT_SETS, F, "gbm"); J["S1 sat-trained"] = (ev, sr.predict(s1, ev, F))
    lv = sr.far_levels(sat, m); prior = dict(far_inl=float(lv.far_inl.mean()), far_n22=float(lv.far_n22.mean()))
    s5, _, st = sr.fit_judge(sat, SAT_SETS, FR, {"rli", "z", "hull", "cover9"}, AUG, lambda x: sr.add_relative(x, prior, 30, m)); del st; gc.collect()
    evr = sr.add_relative(ev, prior, 30, m); J["S5 sat-trained, relative + augmentation"] = (evr, sr.predict(s5, evr, FR))
    pn, _ = cj.judge_probability(ev, NUDT_DESIGN, F, "gbm"); J["reference: NUDT-trained (not shippable)"] = (ev, pn)
    return J, prior
rows, PRIORS, RESULTS = [], {}, {}
for var, m in [("pc", None), ("jetson", 5)]:
    J, PRIORS[var] = judges(m)
    kw = dict(min_choice=3, wide_tau=0.95, override=None, **(dict(jetson=True, m=5, early_tau=0.98) if m else {}))
    base = cj.baseline(D, 8, 11, min_choice=3) if var == "pc" else cj.baseline(D, 8, 8, jetson=True, m=5)
    RESULTS[(var, "base")] = base
    for name, (ev, p) in J.items():
        dmask = ev.data.isin(NUDT_DESIGN).to_numpy()
        tau = cj.ltt_tau(cj.scan_tau(ev[dmask], p[dmask], cj.TAUS, **{k: v for k, v in kw.items() if k != "early_tau"}), NUDT_DESIGN, 0.005)
        tau = cj.NO_ANSWER if tau is None else max(tau, 0.9)
        r = cj.decide(ev, p, tau, **kw); RESULTS[(var, name)] = r
        s = cj.summary(r); a = answer_change(base, r).set_index("data")
        for d in ["official_try"] + NUDT_DESIGN:
            rows.append(dict(var=var, judge=name, cut=tau, data=d, score=s.loc[d, "score"], wrong=int(s.loc[d, "wrong"]), answered=int(s.loc[d, "n_answered"]),
                             queries=s.loc[d, "queries"], gain=cj.paired_gain(base, r, [d])[d], added_right=a.loc[d, "added_right"], added_wrong=a.loc[d, "added_wrong"]))
    sb = cj.summary(base)
    for d in ["official_try"] + NUDT_DESIGN:
        rows.append(dict(var=var, judge="base", cut=None, data=d, score=sb.loc[d, "score"], wrong=int(sb.loc[d, "wrong"]), answered=int(sb.loc[d, "n_answered"]), queries=sb.loc[d, "queries"]))
SJ = pd.DataFrame(rows); SJ.to_csv(R / "it1_sat_judge.csv", index=False)
SJ[SJ.data == "official_try"].drop(columns="data")

In [ ]:
SJ.pivot_table(index=["var", "judge"], columns="data", values=["score", "wrong"]).round(2)

## 3. Is P(right) of the satellite-trained judge honest on thermal photos? (wrong rate by P band)

In [ ]:
bins = [0, 0.5, 0.8, 0.9, 0.95, 0.98, 1.0001]
J, _ = judges(None)
rows = []
for name, (ev, p) in J.items():
    L = ev.assign(p=p)[cj.labelled(ev)]
    L = L.assign(band=pd.cut(L.p, bins, right=False), wrong=L.err_sigma > 8, group=np.where(L.data.isin(NUDT_DESIGN), "NUDT design", L.data))
    g = L.groupby(["group", "band"], observed=True).agg(n=("wrong", "size"), wrong_rate=("wrong", "mean")).reset_index().assign(judge=name)
    rows.append(g)
CAL = pd.concat(rows); CAL.to_csv(R / "it1_calibration.csv", index=False)
CAL.pivot_table(index=["judge", "band"], columns="group", values="wrong_rate").round(3)

## 4. Ranking quality, and the cut on a finer grid near 1

The satellite-trained judges are over-confident on thermal candidates (section 3), so no cut of the standard grid (up to 0.99) passed
Learn-then-Test on NUDT and the rule answered nothing (the T10 'failed calibration -> no answers' fix). Only the ranking matters for a cut, so
the question is whether the ranking separates right from wrong on thermal photos (AUC), and whether a cut higher up the scale passes.

In [ ]:
from sklearn.metrics import roc_auc_score
TAUS_FINE = cj.TAUS + [0.995, 0.998, 0.999, 0.9995, 0.9998, 0.9999]
rows = []
for var, m in [("pc", None), ("jetson", 5)]:
    J, _ = judges(m)
    kw = dict(min_choice=3, wide_tau=0.95, override=None, **(dict(jetson=True, m=5, early_tau=0.98) if m else {}))
    base = RESULTS[(var, "base")]
    for name, (ev, p) in J.items():
        L = cj.labelled(ev).to_numpy()
        auc = {g: round(roc_auc_score((ev.err_sigma <= 4).to_numpy()[L & sel], p[L & sel]), 4) for g, sel in
               [("NUDT design", ev.data.isin(NUDT_DESIGN).to_numpy()), ("official TRY", (ev.data == "official_try").to_numpy())]}
        dmask = ev.data.isin(NUDT_DESIGN).to_numpy()
        kws = {k: v for k, v in kw.items() if k != "early_tau"}
        tau = cj.ltt_tau(cj.scan_tau(ev[dmask], p[dmask], TAUS_FINE, **{**kws, "wide_tau": 0.0}), NUDT_DESIGN, 0.005)
        tau = cj.NO_ANSWER if tau is None else max(tau, 0.9)
        r = cj.decide(ev, p, tau, **{**kw, "wide_tau": tau, "early_tau": max(0.98, tau) if m else None})
        s = cj.summary(r)
        for d in ["official_try"] + NUDT_DESIGN:
            rows.append(dict(var=var, judge=name, auc_nudt=auc["NUDT design"], auc_official_try=auc["official TRY"], cut=tau, data=d,
                             score=s.loc[d, "score"], wrong=int(s.loc[d, "wrong"]), gain=cj.paired_gain(base, r, [d])[d]))
FINE = pd.DataFrame(rows); FINE.to_csv(R / "it1_sat_judge_fine_cut.csv", index=False)
FINE.pivot_table(index=["var", "judge", "auc_nudt", "auc_official_try", "cut"], columns="data", values=["score", "wrong"]).round(2)

## 5. The zero-wrong cut (T7 rule) on the fine grid

Learn-then-Test at alpha 0.005 tolerates a small wrong rate, and at 0.998 the satellite judges keep 1-2 wrong answers on the NUDT design sets
that set the cut. The team's rule is 0 wrong answers on every set, so the cut is instead the smallest one with 0 wrong answers on every NUDT
design set at it and at every stricter cut, plus one grid step (T7 `smallest_safe_tau`, margin 1).

In [ ]:
rows, CUTS0 = [], {}
for var, m in [("pc", None), ("jetson", 5)]:
    J, _ = judges(m)
    base = RESULTS[(var, "base")]
    for name, (ev, p) in J.items():
        if name.startswith("reference"): continue
        kw = dict(min_choice=3, override=None, **(dict(jetson=True, m=5) if m else {}))
        dmask = ev.data.isin(NUDT_DESIGN).to_numpy()
        scan = pd.concat([cj.summary(cj.decide(ev[dmask], p[dmask], t, wide_tau=t, **kw)).reset_index().assign(tau=t) for t in TAUS_FINE])
        tau = cj.smallest_safe_tau(scan, NUDT_DESIGN, 1); CUTS0[(var, name)] = tau
        r = cj.decide(ev, p, tau, wide_tau=tau, early_tau=max(0.98, tau) if m else None, **kw)
        s = cj.summary(r)
        for d in ["official_try"] + NUDT_DESIGN:
            rows.append(dict(var=var, judge=name, cut=tau, data=d, score=s.loc[d, "score"], wrong=int(s.loc[d, "wrong"]), queries=s.loc[d, "queries"],
                             gain=cj.paired_gain(base, r, [d])[d]))
ZW = pd.DataFrame(rows); ZW.to_csv(R / "it1_sat_judge_zero_wrong_cut.csv", index=False)
display(ZW.pivot_table(index=["var", "judge", "cut"], columns="data", values=["score", "wrong"]).round(2))
ZW[ZW.data == "official_try"][["var", "judge", "cut", "score", "wrong", "queries", "gain"]]